# 17 · Loading skewers from the store

The skewers are not computed here. They are read from the **skewer store**, filled from a terminal by
`scripts/production/extract_skewers.py` (layout, identity of a dataset and how to run it:
[`scripts/production/STORE.md`](../scripts/production/STORE.md)). This notebook loads **one** lattice of **one** dataset and checks
that what comes back is what was stored:

1. **Inputs:** the store, the dataset (simulation, snapshot, configuration hash, line of sight) and the lattice
   `N_PER_AXIS`.
2. **Where the lattice comes from:** the lattices are nested, so the $N \times N$ lattice is assembled from the
   skewers of every coarser lattice plus the new ones of each doubling.
3. **Loading** and its checks: shapes, positions on the lattice, the nesting bit for bit.
4. **One skewer:** its transmission.

The statistics of the stored skewers and their convergence with $N$ are in notebook 18.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()                                    # interactive figures in a live notebook
use_style()
from skewerjax.utils.plotting import add_panel_letters
import sys
from skewerjax.utils.paths import PRODUCTION_SCRIPTS_DIR       # the skewer store's reader lives with the production scripts
sys.path.insert(0, str(PRODUCTION_SCRIPTS_DIR))
from skewer_store import SkewerDataset, format_datasets, list_datasets, load_skewers, select_datasets

## 1 · Inputs

- `STORE`: the `--out` of the script (`$SKEWERJAX_STORE`, default `/home/STORAGE/SKEWERS_SkewerJAX`).
- `SIM`, `SNAP`: the simulation and the snapshot (the folders `<STORE>/<SIM>/snap_<SNAP>/`).
- `HASH`: the configuration (the options of $\tau$; the table below lists the hashes). `None` takes the one with the
  largest lattice stored, and it must be unique.
- `AXIS`: the line of sight (1, 2, 3 = $x$, $y$, $z$).
- `N_PER_AXIS`: the lattice to load, $N \times N$ skewers. It must be a power of two and complete in the store
  (`n_max` in the table). Use 16 for a quick check; 256 reads 2 GB.

In [ ]:
STORE = Path(os.environ.get("SKEWERJAX_STORE", "/home/STORAGE/SKEWERS_SkewerJAX"))   # the --out of extract_skewers.py
SIM, SNAP = "TNG50-2", 25
HASH = None                                         # e.g. "cd782fdb19"; None: the configuration with the largest lattice
AXIS = 2                                            # the line of sight: 1, 2, 3 = x, y, z
N_PER_AXIS = 256                                     # the lattice: N_PER_AXIS x N_PER_AXIS skewers
ROWS = list_datasets(STORE)
if not ROWS:
    raise FileNotFoundError(f"no dataset in the store {STORE}: set STORE (or $SKEWERJAX_STORE) to the --out of extract_skewers.py")
print(f"store {STORE}:\n{format_datasets(ROWS)}")

## 2 · The dataset, and where the lattice comes from

A skewer is named by its indices $(i_a, i_b)$ on the finest lattice $N_{\rm ref}$ (4096). The $N \times N$ lattice is
the subset with stride $N_{\rm ref}/N$, so it contains every coarser lattice of the series. The script computes each
doubling's new skewers only ($N^2 - (N/2)^2$ of them), which means **loading the $N \times N$ lattice reads the
skewers of every level $1, 2, 4, \dots, N$**, from the batch files of each. The table gives, per level, the skewers
of this lattice first computed there and the files that hold them.

In [ ]:
SEL = select_datasets(STORE, SIM, SNAP, HASH, axes=(AXIS,))
HASH = SEL["hash"]
DS = SkewerDataset.open(STORE, SIM, SNAP, AXIS, HASH)
if not DS.complete(N_PER_AXIS):
    raise ValueError(f"the {N_PER_AXIS} x {N_PER_AXIS} lattice is not complete in {DS.path}: complete N = {DS.complete_ns()}")
o = DS.config["options"]
print(f"dataset {DS.path}\n  preset {DS.config.get('preset')!r}; lattices complete: N = {DS.complete_ns()}; {len(DS.batch_files())} batch files; "
      f"{len(DS.runs())} run(s)")
print(f"  lines {o['lines']['line_ids']}; {o['pixels']['n_pix']} pixels of {o['pixels']['dv_kms']:.4f} km/s; pieces {o['pieces']['pieces']}; {o['precision']}")
print("  tau: " + ", ".join(f"{k}={v}" for k, v in o["tau"].items()))
SRC = DS.sources(N_PER_AXIS)
print(f"\nthe {N_PER_AXIS} x {N_PER_AXIS} lattice = {' + '.join(str(r['skewers']) for r in SRC)} = {sum(r['skewers'] for r in SRC)} skewers, by level:")
print(f"  {'level N':>8} {'skewers':>8} {'stored':>7}  batch files")
for r in SRC:
    print(f"  {r['n']:8d} {r['skewers']:8d} {r['stored']:7d}  {', '.join(r['files'])}")

## 3 · Loading

`load_skewers` returns the lattice in the order of `core.lattice_columns` (a-major): $\tau$ of every line, the
indices of every skewer, its position, the velocity grid and the dataset's config. Three checks follow. The columns
are the $N \times N$ lattice; the positions are its nodes $x_i = i\,L/N$ on the two transverse axes; and the lattice
$N/2$, loaded on its own, is bit for bit the stride-2 subset of this one (the nesting).

In [ ]:
from skewerjax.core import lattice_columns
t0 = time.perf_counter()
ST = load_skewers(STORE, SIM, SNAP, AXIS, HASH, N_PER_AXIS)
t_load = time.perf_counter() - t0
TAU, V, DV, L, LINE_IDS = ST.tau, ST.v, ST.dv, ST.L, ST.line_ids
print(f"loaded in {t_load:.2f} s: tau {TAU.shape} {TAU.dtype} ({TAU.nbytes / 1e6:.1f} MB) = (lines {list(LINE_IDS)}, {N_PER_AXIS}^2 skewers, pixels)")
print(f"v: {V.size} pixel centres, dv = {DV:.4f} km/s, period {ST.period:.2f} km/s; L = {L:.3f} cMpc, u = {ST.u:.3f} km/s per cMpc; "
      f"d_perp = L / N = {ST.d_perp:.4f} cMpc")
TRANSVERSE = {1: (1, 2), 2: (0, 2), 3: (0, 1)}
a, b = TRANSVERSE[AXIS]
ok_cols = np.array_equal(ST.columns, lattice_columns(N_PER_AXIS, (1, 1)))
ok_pos = np.allclose(ST.positions[:, [a, b]], ST.columns * L / N_PER_AXIS) and np.allclose(ST.positions[:, AXIS - 1], 0.0)
print(f"columns are the {N_PER_AXIS} x {N_PER_AXIS} lattice, a-major: {ok_cols}; positions = (i_a, i_b) L / N, 0 along the line of sight: {ok_pos}")
print(f"tau finite: {bool(np.isfinite(TAU).all())}, >= 0: {bool((TAU >= 0).all())}; max tau per line: "
      + ", ".join(f"{lid} {TAU[j].max():.3g}" for j, lid in enumerate(LINE_IDS)))
if N_PER_AXIS >= 2:
    half = load_skewers(STORE, SIM, SNAP, AXIS, HASH, N_PER_AXIS // 2)
    sub = (ST.columns[:, 0] % 2 == 0) & (ST.columns[:, 1] % 2 == 0)        # the stride-2 subset, in the same order
    print(f"the {N_PER_AXIS // 2} x {N_PER_AXIS // 2} lattice is bit for bit the stride-2 subset of this one: {np.array_equal(half.tau, TAU[:, sub])}")

## 4 · One skewer

The skewer nearest to the middle of the layer $i_b = 0$. The mean transmission that normalises it is $\bar T(z)$ over
the loaded skewers (one snapshot: every pixel at the same $a$, as in notebook 11). The rows show (a) the transmission
of each line and of the total, (b) $\bar T$, and (c) the overfluxes $\delta = T/\bar T - 1$.

In [ ]:
from skewerjax.summary_stats import ensemble_mean_transmissions
from skewerjax.utils.plotting import plot_toy_skewer
T = np.exp(-TAU)[None]                                                   # (1 line of sight, lines, skewers, pixels)
T_BAR = ensemble_mean_transmissions(T, mean="redshift", grid=np.full(V.size, ST.config["snapshot"]["a"]))[0]
on_b0 = np.flatnonzero(ST.columns[:, 1] == 0)
j = int(on_b0[np.argmin(np.abs(ST.columns[on_b0, 0] - N_PER_AXIS // 2))])
fig, axs, _ = plot_toy_skewer(L, None, V, TAU[:, j], mean_transmission=T_BAR[:, j], line_ids=LINE_IDS)
add_panel_letters(axs)
plt.show()
print(f"the skewer (i_a, i_b) = {tuple(ST.columns[j])} along {'xyz'[AXIS - 1]}, at {tuple(np.round(ST.positions[j], 3))} cMpc; "
      "mean transmission over the lattice: " + ", ".join(f"{lid} {np.exp(-TAU[k]).mean():.4f}" for k, lid in enumerate(LINE_IDS)))